<a href="https://colab.research.google.com/github/johrosa/srwi/blob/main/plant_phenology_multisite_pixel_sampling_colab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plant phenology monitoring — multiple GeoPackage points

Google Colab workflow using **Sentinel-2 + ERA5-Land**.

The notebook:
- imports a `.gpkg` point layer;
- processes every point independently;
- extracts NDVI/EVI2/NDMI time series;
- derives SOS, Peak and EOS by site and phenological year;
- extracts ERA5-Land climate by site;
- identifies heat, heavy-rain and dry anomalies;
- compares phenology and climate;
- exports site-level CSV tables.


In [ ]:
!pip -q install earthengine-api geemap geopandas pyogrio scipy

In [ ]:
import ee, geemap
import geopandas as gpd
import pyogrio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from google.colab import files
from IPython.display import display

## 1. Parameters

In [ ]:
PROJECT_ID = "ee-rnrimpact"

SITE_ID_FIELD = "id"
TARGET_GPKG_LAYER = "manga"
REFERENCE_GPKG_LAYER = None  # None = second GeoPackage layer
TARGET_VEGETATION_NAME = "mango_flowering"
REFERENCE_VEGETATION_NAME = "reference_vegetation"

START_DATE = "2018-01-01"
END_DATE = "2025-12-31"

CLIM_BASELINE_START = "2000-01-01"
CLIM_BASELINE_END = "2024-12-31"

PHENO_YEAR_START_MONTH = 7

COMPOSITE_DAYS = 10
CLEAR_THRESHOLD = 0.60
S2_SCALE_M = 20

PHENO_THRESHOLD_FRACTION = 0.20
SMOOTH_WINDOW = 7
SMOOTH_POLYORDER = 2

HEAT_PERCENTILE = 95
HEAVY_RAIN_PERCENTILE = 95
ROLLING_RAIN_DAYS = 30
DRY_Z_THRESHOLD = -1.5

ERA5_SCALE_M = 11132

CYCLONE_EVENTS = [
    {"date": "2022-02-05", "name": "Example cyclone/event"}
]

## 2. Earth Engine authentication

In [ ]:
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized")

Earth Engine initialized


## 3. Upload GeoPackage

In [ ]:
uploaded = files.upload()

gpkg_files = [f for f in uploaded if f.lower().endswith(".gpkg")]
if not gpkg_files:
    raise ValueError("Upload a .gpkg file")

GPKG_FILE = gpkg_files[0]
layers = pyogrio.list_layers(GPKG_FILE)

print("GeoPackage:", GPKG_FILE)
display(pd.DataFrame(layers, columns=["layer", "geometry_type"]))

Saving srwi_maroa_manga.gpkg to srwi_maroa_manga.gpkg
GeoPackage: srwi_maroa_manga.gpkg


/usr/local/lib/python3.12/dist-packages/pyogrio/core.py:169: RuntimeWarning: GPKG: bad application_id=0x00000000 on 'srwi_maroa_manga.gpkg'
  return ogr_list_layers(get_vsi_path_or_buffer(path_or_buffer))


,layer,geometry_type
0,manga,Point Z
1,hafa,Point Z
2,android_metadata,None


In [ ]:
layer_names = [layer[0] for layer in layers]

if TARGET_GPKG_LAYER is None:
    TARGET_GPKG_LAYER = layer_names[0]

if TARGET_GPKG_LAYER not in layer_names:
    raise ValueError(f"Target layer '{TARGET_GPKG_LAYER}' not found in GeoPackage")

if REFERENCE_GPKG_LAYER is None:
    reference_candidates = [name for name in layer_names if name != TARGET_GPKG_LAYER]
    REFERENCE_GPKG_LAYER = reference_candidates[0] if reference_candidates else None

if REFERENCE_GPKG_LAYER is not None and REFERENCE_GPKG_LAYER not in layer_names:
    raise ValueError(f"Reference layer '{REFERENCE_GPKG_LAYER}' not found in GeoPackage")


def read_site_layer(layer_name, vegetation_role, vegetation_name, id_prefix):
    layer_gdf = gpd.read_file(GPKG_FILE, layer=layer_name)

    if layer_gdf.crs is None:
        raise ValueError(f"GeoPackage layer '{layer_name}' has no CRS")

    layer_gdf = layer_gdf[layer_gdf.geometry.notna() & ~layer_gdf.geometry.is_empty].copy()
    layer_gdf = layer_gdf.explode(index_parts=False, ignore_index=True)

    if not set(layer_gdf.geometry.geom_type).issubset({"Point"}):
        raise ValueError(f"Layer '{layer_name}' must contain Point/MultiPoint geometries only")

    layer_gdf = layer_gdf.to_crs("EPSG:4326")

    if SITE_ID_FIELD in layer_gdf.columns:
        raw_id = layer_gdf[SITE_ID_FIELD].astype(str)
    else:
        raw_id = pd.Series([f"{id_prefix}{i+1:03d}" for i in range(len(layer_gdf))])

    layer_gdf["site_id"] = vegetation_role + "_" + raw_id.astype(str)
    if layer_gdf["site_id"].duplicated().any():
        layer_gdf["site_id"] = layer_gdf["site_id"] + "_" + (layer_gdf.groupby("site_id").cumcount()+1).astype(str)

    layer_gdf["source_layer"] = layer_name
    layer_gdf["vegetation_role"] = vegetation_role
    layer_gdf["vegetation_name"] = vegetation_name
    layer_gdf["longitude"] = layer_gdf.geometry.x
    layer_gdf["latitude"] = layer_gdf.geometry.y

    return layer_gdf


target_gdf = read_site_layer(
    TARGET_GPKG_LAYER,
    vegetation_role="target",
    vegetation_name=TARGET_VEGETATION_NAME,
    id_prefix="M"
)

reference_gdf = read_site_layer(
    REFERENCE_GPKG_LAYER,
    vegetation_role="reference",
    vegetation_name=REFERENCE_VEGETATION_NAME,
    id_prefix="R"
) if REFERENCE_GPKG_LAYER is not None else gpd.GeoDataFrame(columns=target_gdf.columns, crs=target_gdf.crs)

gdf = pd.concat([target_gdf, reference_gdf], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")

print("Target layer:", TARGET_GPKG_LAYER, "sites:", len(target_gdf))
print("Reference layer:", REFERENCE_GPKG_LAYER, "sites:", len(reference_gdf))
print("Total sites:", len(gdf))
display(gdf[["site_id", "vegetation_role", "vegetation_name", "source_layer", "longitude", "latitude"]].head(30))

## 4. Convert sites to Earth Engine

In [ ]:
features = []
for _, r in gdf.iterrows():
    features.append(
        ee.Feature(
            ee.Geometry.Point([float(r.longitude), float(r.latitude)]),
            {
                "site_id": str(r.site_id),
                "vegetation_role": str(r.vegetation_role),
                "vegetation_name": str(r.vegetation_name),
                "source_layer": str(r.source_layer),
                "longitude": float(r.longitude),
                "latitude": float(r.latitude)
            }
        )
    )

sites = ee.FeatureCollection(features)
target_sites = sites.filter(ee.Filter.eq("vegetation_role", "target"))
reference_sites = sites.filter(ee.Filter.eq("vegetation_role", "reference"))
study_geometry = sites.geometry().bounds().buffer(1000)

print("EE sites:", sites.size().getInfo())
print("Target EE sites:", target_sites.size().getInfo())
print("Reference EE sites:", reference_sites.size().getInfo())

Map = geemap.Map()
Map.centerObject(sites, 7)
Map.addLayer(target_sites, {"color":"red"}, "Target mango flowering sites")
Map.addLayer(reference_sites, {"color":"blue"}, "Reference vegetation sites")
Map

## 5. Sentinel-2 preprocessing

**Lightweight mode:** Sentinel‑2 values are sampled from the single pixel intersecting each point. No buffer and no neighborhood median are calculated.

In [ ]:
S2_ID = "COPERNICUS/S2_SR_HARMONIZED"
CS_ID = "GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED"

s2 = (ee.ImageCollection(S2_ID)
      .filterBounds(study_geometry)
      .filterDate(START_DATE, ee.Date(END_DATE).advance(1,"day")))

cs = (ee.ImageCollection(CS_ID)
      .filterBounds(study_geometry)
      .filterDate(START_DATE, ee.Date(END_DATE).advance(1,"day")))

s2_linked = s2.linkCollection(cs, ["cs_cdf"])

def prep_s2(img):
    clear = img.select("cs_cdf").gte(CLEAR_THRESHOLD)
    x = img.select(["B4","B8","B11"]).multiply(0.0001)

    red = x.select("B4")
    nir = x.select("B8")
    swir = x.select("B11")

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    evi2 = nir.subtract(red).multiply(2.5).divide(
        nir.add(red.multiply(2.4)).add(1)
    ).rename("EVI2")
    ndmi = nir.subtract(swir).divide(nir.add(swir)).rename("NDMI")

    return ee.Image.cat([ndvi,evi2,ndmi]).updateMask(clear).copyProperties(
        img, ["system:time_start","system:index"]
    )

s2_vi = s2_linked.map(prep_s2)
print("Sentinel-2 scenes:", s2_vi.size().getInfo())

Sentinel-2 scenes: 3761


## 6. Sample Sentinel-2 by site before compositing

In [ ]:
import time

start_ee = ee.Date(START_DATE)
end_ee = ee.Date(END_DATE).advance(1,"day")
n_steps = end_ee.difference(start_ee,"day").divide(COMPOSITE_DAYS).ceil().int()


def date_chunks(start_date, end_date, freq="YS"):
    start = pd.Timestamp(start_date)
    end_exclusive = pd.Timestamp(end_date) + pd.Timedelta(days=1)
    breaks = pd.date_range(start.normalize(), end_exclusive, freq=freq)
    bounds = sorted(set([start, end_exclusive, *breaks]))

    for d0, d1 in zip(bounds[:-1], bounds[1:]):
        d0 = max(d0, start)
        d1 = min(d1, end_exclusive)
        if d0 < d1:
            yield d0, d1


def collection_region_df(collection, point, scale):
    rows = collection.sort("system:time_start").getRegion(point, scale=scale).getInfo()
    if len(rows) <= 1:
        return pd.DataFrame()
    return pd.DataFrame(rows[1:], columns=rows[0])


def sample_s2_site(site, pause_seconds=0.2, chunk_freq="YS"):
    point = ee.Geometry.Point([float(site["longitude"]), float(site["latitude"])])
    parts = []

    for d0, d1 in date_chunks(START_DATE, END_DATE, freq=chunk_freq):
        collection = s2_vi.filterBounds(point).filterDate(
            d0.strftime("%Y-%m-%d"),
            d1.strftime("%Y-%m-%d")
        )
        df = collection_region_df(collection, point, S2_SCALE_M)
        if not df.empty:
            parts.append(df)

        if pause_seconds:
            time.sleep(pause_seconds)

    if not parts:
        return pd.DataFrame()

    df = pd.concat(parts, ignore_index=True)
    df = df.rename(columns={"id": "image_id", "time": "time_ms"})
    df["site_id"] = site["site_id"]
    df["vegetation_role"] = site["vegetation_role"]
    df["vegetation_name"] = site["vegetation_name"]
    df["source_layer"] = site["source_layer"]
    df["longitude"] = float(site["longitude"])
    df["latitude"] = float(site["latitude"])
    df["date"] = pd.to_datetime(df["time_ms"], unit="ms")

    return df[[
        "site_id", "vegetation_role", "vegetation_name", "source_layer",
        "longitude", "latitude", "date", "image_id",
        "NDVI", "EVI2", "NDMI"
    ]]

print("Composite windows:", n_steps.getInfo())
print("Sampling will run site by site in the next cell.")

## 7. Build 10-day point composites

In [ ]:
site_records = gdf[[
    "site_id", "vegetation_role", "vegetation_name", "source_layer",
    "longitude", "latitude"
]].to_dict("records")
sample_parts = []

for site in site_records:
    part = sample_s2_site(site)
    print(f"{site['site_id']}: {len(part)} Sentinel-2 point samples")
    if not part.empty:
        sample_parts.append(part)

s2_raw_df = pd.concat(sample_parts, ignore_index=True) if sample_parts else pd.DataFrame()
print("Raw point samples:", len(s2_raw_df))
display(s2_raw_df.head(20))

In [ ]:
vi_df = s2_raw_df.copy()

if vi_df.empty:
    raise ValueError("No valid Sentinel-2 point samples were returned.")

vi_df["date"] = pd.to_datetime(vi_df["date"])

for c in ["NDVI", "EVI2", "NDMI", "longitude", "latitude"]:
    vi_df[c] = pd.to_numeric(vi_df[c], errors="coerce")

start_ts = pd.Timestamp(START_DATE)
vi_df["composite_index"] = ((vi_df["date"] - start_ts).dt.days // COMPOSITE_DAYS).astype(int)
vi_df["date"] = start_ts + pd.to_timedelta(
    vi_df["composite_index"] * COMPOSITE_DAYS,
    unit="D"
)

vi_df = (
    vi_df
    .groupby([
        "site_id", "vegetation_role", "vegetation_name", "source_layer",
        "longitude", "latitude", "date"
    ], as_index=False)
    .agg(
        n_images=("NDVI", "count"),
        NDVI=("NDVI", "median"),
        EVI2=("EVI2", "median"),
        NDMI=("NDMI", "median")
    )
    .sort_values(["site_id", "date"])
    .reset_index(drop=True)
)

vegetation_timeseries_summary = (
    vi_df
    .groupby(["vegetation_role", "vegetation_name", "source_layer", "date"], as_index=False)
    .agg(
        site_count=("site_id", "nunique"),
        mean_NDVI=("NDVI", "mean"),
        median_NDVI=("NDVI", "median"),
        mean_EVI2=("EVI2", "mean"),
        mean_NDMI=("NDMI", "mean")
    )
)

reference_vi_df = vi_df[vi_df["vegetation_role"] == "reference"].copy()

display(vi_df.head(20))
display(vegetation_timeseries_summary.head(20))
print("10-day point composites:", len(vi_df))

## 8. Phenology by site

In [ ]:
def assign_pheno_year(dates, start_month):
    dates = pd.DatetimeIndex(dates)
    return np.where(dates.month >= start_month, dates.year, dates.year-1)

vi_df["pheno_year"] = assign_pheno_year(
    vi_df["date"], PHENO_YEAR_START_MONTH
)

def safe_savgol(y, window=7, polyorder=2):
    y = np.asarray(y,dtype=float)
    n = len(y)
    if n < 5:
        return y
    w = min(window, n if n%2 else n-1)
    if w < 5:
        return y
    if w%2 == 0:
        w -= 1
    return savgol_filter(y, w, min(polyorder,w-2), mode="interp")

def phenology_for_season(group):
    g = group[["date","NDVI"]].dropna().sort_values("date")
    if len(g) < 5:
        return None, None

    idx = pd.date_range(g.date.min(), g.date.max(), freq="D")
    s = (g.set_index("date")["NDVI"]
         .reindex(idx)
         .interpolate("time")
         .ffill().bfill())

    sm = pd.Series(
        safe_savgol(s.values, SMOOTH_WINDOW, SMOOTH_POLYORDER),
        index=idx
    )

    vmin = float(sm.min())
    vmax = float(sm.max())
    amp = vmax-vmin
    if amp <= 0:
        return None, None

    threshold = vmin + PHENO_THRESHOLD_FRACTION*amp
    peak = sm.idxmax()
    pos = sm.index.get_loc(peak)

    pre = sm.iloc[:pos+1]
    post = sm.iloc[pos:]

    sos_vals = pre[pre >= threshold]
    eos_vals = post[post >= threshold]

    sos = sos_vals.index[0] if len(sos_vals) else pd.NaT
    eos = eos_vals.index[-1] if len(eos_vals) else pd.NaT

    metrics = {
        "SOS":sos,
        "Peak":peak,
        "EOS":eos,
        "SOS_DOY":sos.dayofyear if pd.notna(sos) else np.nan,
        "Peak_DOY":peak.dayofyear,
        "EOS_DOY":eos.dayofyear if pd.notna(eos) else np.nan,
        "NDVI_min":vmin,
        "NDVI_peak":vmax,
        "Amplitude":amp,
        "Season_length_days":(eos-sos).days if pd.notna(sos) and pd.notna(eos) else np.nan,
        "NDVI_integral":float(np.trapz(sm.values,dx=1))
    }

    curve = pd.DataFrame({"date":idx,"NDVI_smooth":sm.values})
    return metrics,curve

In [ ]:
rows = []
curves = []
flowering_vi_df = vi_df[vi_df["vegetation_role"] == "target"].copy()

for (site_id, yr), grp in flowering_vi_df.groupby(["site_id", "pheno_year"]):
    metrics, curve = phenology_for_season(grp)
    if metrics is None:
        continue

    first = grp.iloc[0]
    metrics["site_id"] = site_id
    metrics["vegetation_role"] = first["vegetation_role"]
    metrics["vegetation_name"] = first["vegetation_name"]
    metrics["source_layer"] = first["source_layer"]
    metrics["phenology_target"] = "flowering_proxy"
    metrics["pheno_year"] = int(yr)
    rows.append(metrics)

    curve["site_id"] = site_id
    curve["vegetation_role"] = first["vegetation_role"]
    curve["vegetation_name"] = first["vegetation_name"]
    curve["source_layer"] = first["source_layer"]
    curve["phenology_target"] = "flowering_proxy"
    curve["pheno_year"] = int(yr)
    curves.append(curve)

phenology_df = pd.DataFrame(rows)
if not phenology_df.empty:
    phenology_df = phenology_df.sort_values(["site_id", "pheno_year"]).reset_index(drop=True)

smooth_df = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()

print("Mango flowering proxy seasons:", len(phenology_df))
display(phenology_df.head(30))

## 9. Plot phenology

In [ ]:
plot_site_ids = flowering_vi_df["site_id"].unique()[:5]

for site_id in plot_site_ids:
    obs = flowering_vi_df[flowering_vi_df.site_id == site_id]
    sm = smooth_df[smooth_df.site_id == site_id]
    met = phenology_df[phenology_df.site_id == site_id]

    plt.figure(figsize=(14,5))
    plt.scatter(obs.date,obs.NDVI,s=12,alpha=.4,label="NDVI")
    plt.plot(sm.date,sm.NDVI_smooth,lw=2,label="Smoothed")

    for _,r in met.iterrows():
        if pd.notna(r.SOS): plt.axvline(r.SOS,ls="--",alpha=.3,label="Flowering window start")
        if pd.notna(r.Peak): plt.scatter(r.Peak,r.NDVI_peak,marker="^",s=50,label="Flowering proxy peak")
        if pd.notna(r.EOS): plt.axvline(r.EOS,ls=":",alpha=.3,label="Flowering window end")

    plt.title(f"Mango flowering proxy - {site_id}")
    plt.ylabel("NDVI")
    plt.grid(alpha=.25)
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys())
    plt.show()

## 10. ERA5-Land daily climate by site

In [ ]:
import time

era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
         .filterBounds(study_geometry)
         .filterDate(CLIM_BASELINE_START,ee.Date(END_DATE).advance(1,"day"))
         .select([
             "temperature_2m",
             "temperature_2m_min",
             "temperature_2m_max",
             "total_precipitation_sum"
         ]))

def sample_era5_site(site, pause_seconds=0.2, chunk_freq="YS"):
    point = ee.Geometry.Point([float(site["longitude"]), float(site["latitude"])])
    parts = []

    for d0, d1 in date_chunks(CLIM_BASELINE_START, END_DATE, freq=chunk_freq):
        collection = era5.filterBounds(point).filterDate(
            d0.strftime("%Y-%m-%d"),
            d1.strftime("%Y-%m-%d")
        )
        df = collection_region_df(collection, point, ERA5_SCALE_M)
        if not df.empty:
            parts.append(df)

        if pause_seconds:
            time.sleep(pause_seconds)

    if not parts:
        return pd.DataFrame()

    df = pd.concat(parts, ignore_index=True)
    df = df.rename(columns={
        "time": "time_ms",
        "temperature_2m": "Tmean_K",
        "temperature_2m_min": "Tmin_K",
        "temperature_2m_max": "Tmax_K",
        "total_precipitation_sum": "P_m"
    })
    df["site_id"] = site["site_id"]
    df["vegetation_role"] = site["vegetation_role"]
    df["vegetation_name"] = site["vegetation_name"]
    df["source_layer"] = site["source_layer"]
    df["date"] = pd.to_datetime(df["time_ms"], unit="ms")

    return df[[
        "site_id", "vegetation_role", "vegetation_name", "source_layer",
        "date", "Tmean_K", "Tmin_K", "Tmax_K", "P_m"
    ]]

print("ERA5-Land daily images:", era5.size().getInfo())
print("ERA5 sampling will run site by site in the next cell.")

In [ ]:
site_records = gdf[[
    "site_id", "vegetation_role", "vegetation_name", "source_layer",
    "longitude", "latitude"
]].to_dict("records")

climate_parts = []
for site in site_records:
    part = sample_era5_site(site)
    print(f"{site['site_id']}: {len(part)} ERA5 daily samples")
    if not part.empty:
        climate_parts.append(part)

clim_df = pd.concat(climate_parts, ignore_index=True) if climate_parts else pd.DataFrame()
if clim_df.empty:
    raise ValueError("No valid ERA5-Land samples were returned.")

clim_df["date"] = pd.to_datetime(clim_df["date"])

for c in ["Tmean_K","Tmin_K","Tmax_K","P_m"]:
    clim_df[c] = pd.to_numeric(clim_df[c],errors="coerce")

clim_df["Tmean_C"] = clim_df.Tmean_K-273.15
clim_df["Tmin_C"] = clim_df.Tmin_K-273.15
clim_df["Tmax_C"] = clim_df.Tmax_K-273.15
clim_df["P_mm"] = (clim_df.P_m*1000).clip(lower=0)

clim_df["doy"] = clim_df.date.dt.dayofyear
clim_df["month"] = clim_df.date.dt.month
clim_df["pheno_year"] = assign_pheno_year(
    clim_df.date, PHENO_YEAR_START_MONTH
)

clim_df = clim_df.sort_values(["site_id","date"]).reset_index(drop=True)
display(clim_df.head())

## 11. Climate extremes by site

In [ ]:
parts = []

for site_id,sdf in clim_df.groupby("site_id"):
    sdf = sdf.sort_values("date").copy()

    base = sdf[
        (sdf.date >= pd.Timestamp(CLIM_BASELINE_START)) &
        (sdf.date <= pd.Timestamp(CLIM_BASELINE_END))
    ].copy()

    if base.empty:
        continue

    tq = base.groupby("doy").Tmax_C.quantile(HEAT_PERCENTILE/100)
    pq = base.groupby("doy").P_mm.quantile(HEAVY_RAIN_PERCENTILE/100)

    sdf["Tmax_q"] = sdf.doy.map(tq)
    sdf["P_q"] = sdf.doy.map(pq)

    sdf["extreme_heat"] = sdf.Tmax_C > sdf.Tmax_q
    sdf["heavy_rain"] = sdf.P_mm > sdf.P_q

    sdf["P_roll_mm"] = (
        sdf.set_index("date").P_mm
        .rolling(f"{ROLLING_RAIN_DAYS}D",
                 min_periods=max(10,ROLLING_RAIN_DAYS//2))
        .sum().values
    )

    b = sdf[
        (sdf.date >= pd.Timestamp(CLIM_BASELINE_START)) &
        (sdf.date <= pd.Timestamp(CLIM_BASELINE_END))
    ]

    stats = b.groupby("month").P_roll_mm.agg(["mean","std"])
    sdf = sdf.join(stats,on="month",rsuffix="_base")

    sdf["P30_z"] = (sdf.P_roll_mm-sdf["mean"])/sdf["std"].replace(0,np.nan)
    sdf["dry_extreme"] = sdf.P30_z <= DRY_Z_THRESHOLD

    parts.append(sdf)

clim_ext_df = pd.concat(parts,ignore_index=True)

analysis_clim = clim_ext_df[
    (clim_ext_df.date >= pd.Timestamp(START_DATE)) &
    (clim_ext_df.date <= pd.Timestamp(END_DATE))
].copy()

display(analysis_clim.head())

## 12. Site × year climate summary and phenology merge

In [ ]:
season_climate = (
    analysis_clim
    .groupby(["site_id", "vegetation_role", "vegetation_name", "source_layer", "pheno_year"])
    .agg(
        mean_T_C=("Tmean_C","mean"),
        max_T_C=("Tmax_C","max"),
        total_P_mm=("P_mm","sum"),
        heat_days=("extreme_heat","sum"),
        heavy_rain_days=("heavy_rain","sum"),
        dry_extreme_days=("dry_extreme","sum"),
        min_P30_z=("P30_z","min")
    )
    .reset_index()
)

season_climate["pheno_year"] = season_climate.pheno_year.astype(int)

flowering_climate_df = phenology_df.merge(
    season_climate,
    on=["site_id", "vegetation_role", "vegetation_name", "source_layer", "pheno_year"],
    how="left"
)

reference_season_climate = season_climate[
    season_climate["vegetation_role"] == "reference"
].copy()

combined_df = flowering_climate_df

flowering_rain_columns = [
    "NDVI_peak", "Amplitude", "NDVI_integral",
    "total_P_mm", "heavy_rain_days", "dry_extreme_days", "heat_days"
]
flowering_rain_corr = pd.DataFrame()
if len(flowering_climate_df) >= 3:
    flowering_rain_corr = flowering_climate_df[flowering_rain_columns].corr()

print("Flowering/rain/cyclone hypothesis table rows:", len(flowering_climate_df))
display(flowering_climate_df.head(30))
display(reference_season_climate.head(30))
display(flowering_rain_corr)

## 13. Event-centred NDVI response

In [1]:
PRE_EVENT_DAYS = 90
POST_EVENT_DAYS = [30, 90]

event_rows = []

for event in CYCLONE_EVENTS:
    event_date = pd.Timestamp(event["date"])
    event_name = event.get("name", "event")

    for site_id, sdf in flowering_vi_df.groupby("site_id"):
        first = sdf.iloc[0]
        pre = sdf[
            (sdf.date >= event_date - pd.Timedelta(days=PRE_EVENT_DAYS)) &
            (sdf.date < event_date)
        ].NDVI.dropna()

        row = {
            "site_id": site_id,
            "vegetation_role": first["vegetation_role"],
            "vegetation_name": first["vegetation_name"],
            "source_layer": first["source_layer"],
            "event_name": event_name,
            "event_date": event_date,
            "pre_NDVI": pre.mean()
        }

        for days in POST_EVENT_DAYS:
            post = sdf[
                (sdf.date >= event_date) &
                (sdf.date <= event_date + pd.Timedelta(days=days))
            ].NDVI.dropna()
            row[f"post{days}_NDVI"] = post.mean()
            row[f"delta{days}"] = post.mean() - pre.mean()

        event_rows.append(row)

event_summary_df = pd.DataFrame(event_rows)
display(event_summary_df)

NameError: name 'CYCLONE_EVENTS' is not defined

## 14. Export CSV results

In [ ]:
target_gdf.drop(columns="geometry").to_csv("target_mango_sites_from_geopackage.csv",index=False)
reference_gdf.drop(columns="geometry").to_csv("reference_vegetation_sites_from_geopackage.csv",index=False)
gdf.drop(columns="geometry").to_csv("sites_from_geopackage.csv",index=False)
vi_df.to_csv("sentinel2_timeseries_by_site.csv",index=False)
vegetation_timeseries_summary.to_csv("sentinel2_timeseries_summary_by_vegetation_role.csv",index=False)
flowering_vi_df.to_csv("sentinel2_timeseries_mango_flowering_sites.csv",index=False)
reference_vi_df.to_csv("sentinel2_timeseries_reference_vegetation_sites.csv",index=False)
phenology_df.to_csv("mango_flowering_proxy_metrics_by_site.csv",index=False)
smooth_df.to_csv("mango_flowering_proxy_smooth_curves.csv",index=False)
analysis_clim.to_csv("climate_extremes_by_site.csv",index=False)
season_climate.to_csv("climate_summary_by_site_year.csv",index=False)
reference_season_climate.to_csv("reference_climate_summary_by_site_year.csv",index=False)
flowering_climate_df.to_csv("mango_flowering_climate_by_site_year.csv",index=False)
flowering_rain_corr.to_csv("mango_flowering_rain_correlation.csv")
combined_df.to_csv("phenology_climate_by_site_year.csv",index=False)
event_summary_df.to_csv("cyclone_event_response_mango_flowering_sites.csv",index=False)

print("Exports completed.")

## Scaling note

This version samples Sentinel-2 and ERA5-Land site by site before doing tabular aggregation in pandas. For hundreds or thousands of sites, process sites in batches or export per-site tables to Drive/Cloud Storage.
